# Plant Leaf Disease Classification Using Computer Vision and Machine Learning

This notebook contains the main implementation for a Computer Vision final project focused on classifying plant leaf images into healthy and diseased categories.

The project uses selected leaf image classes from the New Plant Diseases Dataset, specifically Peach, Pepper Bell, and Strawberry leaves. The pipeline includes dataset loading, preprocessing, disease spot segmentation, feature extraction, traditional threshold-based classification, machine learning classification, and model evaluation.

## Library & Data Checking

In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")

Libraries imported successfully.


In [8]:
# ================================
# Global Configuration
# ================================

PROJECT_ROOT = Path.cwd()
DATASET_DIR = PROJECT_ROOT / "data" / "raw"

SELECTED_CLASSES = [
    "Peach___Bacterial_spot",
    "Peach___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Strawberry___Leaf_scorch",
    "Strawberry___healthy"
]

IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png"]

RANDOM_STATE = 42
TEST_SIZE = 0.2

print("Project root:", PROJECT_ROOT)
print("Dataset directory:", DATASET_DIR)

Project root: c:\not_default\Projects\plant-leaf-disease-classification
Dataset directory: c:\not_default\Projects\plant-leaf-disease-classification\data\raw


In [9]:
# ================================
# Check Dataset Directory
# ================================

if not DATASET_DIR.exists():
    raise FileNotFoundError(f"Dataset directory not found: {DATASET_DIR}")

available_folders = sorted([folder.name for folder in DATASET_DIR.iterdir() if folder.is_dir()])

print("Available folders in data/raw:")
for folder in available_folders:
    print("-", folder)

missing_classes = [class_name for class_name in SELECTED_CLASSES if class_name not in available_folders]

if missing_classes:
    print("\nMissing selected class folders:")
    for class_name in missing_classes:
        print("-", class_name)
    raise FileNotFoundError("Some selected class folders are missing. Please check folder names.")
else:
    print("\nAll selected class folders are available.")

Available folders in data/raw:
- Peach___Bacterial_spot
- Peach___healthy
- Pepper,_bell___Bacterial_spot
- Pepper,_bell___healthy
- Strawberry___Leaf_scorch
- Strawberry___healthy

All selected class folders are available.


In [10]:
# ================================
# Helper Function
# ================================

def parse_class_name(class_name):
    """
    Convert original folder name into plant type, disease name, and binary label.

    Example:
    Peach___Bacterial_spot -> plant_type: Peach, disease_name: Bacterial spot, label: diseased
    Peach___healthy        -> plant_type: Peach, disease_name: healthy, label: healthy
    """
    plant_type, disease_name = class_name.split("___")

    plant_type = plant_type.replace(",", "").replace("_", " ").strip()
    disease_name = disease_name.replace("_", " ").strip()

    if disease_name.lower() == "healthy":
        label = "healthy"
    else:
        label = "diseased"

    return plant_type, disease_name, label

In [11]:
# ================================
# Build Image Dataframe
# ================================

image_records = []

for class_name in SELECTED_CLASSES:
    class_dir = DATASET_DIR / class_name

    for image_path in class_dir.iterdir():
        if image_path.suffix.lower() in IMAGE_EXTENSIONS:
            plant_type, disease_name, label = parse_class_name(class_name)

            image_records.append({
                "image_path": str(image_path),
                "class_name": class_name,
                "plant_type": plant_type,
                "disease_name": disease_name,
                "label": label
            })

df_images = pd.DataFrame(image_records)

print("Total images loaded:", len(df_images))
df_images.head()

Total images loaded: 11065


,image_path,class_name,plant_type,disease_name,label
0,c:\not_default\Projects\plant-leaf-disease-cla...,Peach___Bacterial_spot,Peach,Bacterial spot,diseased
1,c:\not_default\Projects\plant-leaf-disease-cla...,Peach___Bacterial_spot,Peach,Bacterial spot,diseased
2,c:\not_default\Projects\plant-leaf-disease-cla...,Peach___Bacterial_spot,Peach,Bacterial spot,diseased
3,c:\not_default\Projects\plant-leaf-disease-cla...,Peach___Bacterial_spot,Peach,Bacterial spot,diseased
4,c:\not_default\Projects\plant-leaf-disease-cla...,Peach___Bacterial_spot,Peach,Bacterial spot,diseased


In [12]:
# ================================
# Dataset Summary
# ================================

print("Image count by original class:")
display(df_images["class_name"].value_counts().sort_index())

print("\nImage count by binary label:")
display(df_images["label"].value_counts())

print("\nImage count by plant type and label:")
display(pd.crosstab(df_images["plant_type"], df_images["label"]))

Image count by original class:


class_name
Peach___Bacterial_spot           1838
Peach___healthy                  1728
Pepper,_bell___Bacterial_spot    1913
Pepper,_bell___healthy           1988
Strawberry___Leaf_scorch         1774
Strawberry___healthy             1824
Name: count, dtype: int64


Image count by binary label:


label
healthy     5540
diseased    5525
Name: count, dtype: int64


Image count by plant type and label:


label,diseased,healthy
plant_type,,
Peach,1838,1728
Pepper bell,1913,1988
Strawberry,1774,1824


In [13]:
# ================================
# Train-Test Split
# ================================

train_df, test_df = train_test_split(
    df_images,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df_images["label"]
)

print("Training data:", len(train_df))
print("Testing data:", len(test_df))

print("\nTraining label distribution:")
display(train_df["label"].value_counts())

print("\nTesting label distribution:")
display(test_df["label"].value_counts())

Training data: 8852
Testing data: 2213

Training label distribution:


label
healthy     4432
diseased    4420
Name: count, dtype: int64


Testing label distribution:


label
healthy     1108
diseased    1105
Name: count, dtype: int64

In [14]:
# ================================
# Save Dataset Index
# ================================

features_dir = PROJECT_ROOT / "features"
features_dir.mkdir(exist_ok=True)

df_images.to_csv(features_dir / "image_dataset_index.csv", index=False)
train_df.to_csv(features_dir / "train_image_index.csv", index=False)
test_df.to_csv(features_dir / "test_image_index.csv", index=False)

print("Dataset index files saved to features/")
print("- image_dataset_index.csv")
print("- train_image_index.csv")
print("- test_image_index.csv")

Dataset index files saved to features/
- image_dataset_index.csv
- train_image_index.csv
- test_image_index.csv
